# 07 — Appeal Overturn Risk Model (Optional / Illustrative)
## Prior Authorization Intelligence System (PAIS)
**Phase 5 | Predictive Modeling and Model Explainability**

---

> **⚠ ILLUSTRATIVE MODEL — SMALL DATASET WARNING**  
> This notebook uses 175 rows (the full appeals dataset). Results should be  
> interpreted as a **methodology demonstration only**, not as production-ready  
> model performance. High variance across cross-validation folds is expected  
> and documented. This model would require substantially more data (1,000+ rows minimum)  
> before deployment consideration.

> **Model framing:** This model does **not** approve or deny care. It is a workflow  
> decision-support tool that identifies denied cases at elevated risk of appeal overturn,  
> enabling medical directors and denial management staff to apply additional review  
> **before the final denial letter is issued**.

---

### Analytical Purpose
KFF 2024 data: 79.4% of Medicare Advantage PA appeals result in at least partial  
overturn. This high overturn rate suggests a substantial share of initial denials  
are operationally preventable. If a model can identify which denied requests are  
likely to be overturned on appeal, the medical director can apply an additional  
review layer before finalizing the denial — potentially avoiding an avoidable  
appeals cycle.

### Target Variable
`overturned_flag = 1` if `appeal_outcome` in {Overturned, Partially Overturned}
- Positive rate: **79.4%** — this is an INVERTED imbalance (majority is overturned)
- 175 total appeals; 139 overturned (Overturned: 112, Partially Overturned: 27), 36 upheld

### Dataset Constraint
Only **175 rows** are available (denied requests that reached appeal). For this reason:
- No train/test split is performed (too few rows for meaningful hold-out evaluation)
- 5-fold stratified cross-validation is used instead
- Only LR and Decision Tree are used (no complex models with 175 rows)
- Results are interpreted as proof-of-concept, not production performance

### Additional Feature Available for Appeal Model
`additional_documentation_submitted` (from `fact_appeal`) — whether the appellant  
submitted additional documentation with the appeal request. This is available at  
appeal intake and is not leakage.


## 1. Setup and Data Loading

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import json
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (roc_auc_score, precision_score, recall_score,
                              f1_score, average_precision_score, make_scorer)

BASE = "/sessions/dreamy-cool-ramanujan/mnt/Prior Authorization Intelligence System/"
OUT  = "/sessions/dreamy-cool-ramanujan/mnt/outputs/"

fa     = pd.read_csv(BASE + "prior_auth_requests.csv")
appeal = pd.read_csv(BASE + "appeals.csv")
prov   = pd.read_csv(BASE + "providers.csv")
memb   = pd.read_csv(BASE + "members.csv")
svc    = pd.read_csv(BASE + "services.csv")

print(f"Appeals table: {len(appeal)} rows")
print(f"\nAppeal outcome distribution:")
print(appeal['appeal_outcome'].value_counts())


Appeals table: 175 rows

Appeal outcome distribution:
appeal_outcome
Overturned              112
Upheld                   36
Partially Overturned     27
Name: count, dtype: int64


## 2. Merge and Target Variable

In [ ]:
df = appeal.merge(
    fa[['request_id','request_type','submission_channel','service_id',
        'member_id','provider_id','documentation_complete',
        'estimated_cost','previous_denial_history','auto_eligible',
        'clinical_review_required','submitted_day_of_week']],
    on='request_id', how='left'
)
df = df.merge(prov[['provider_id','provider_type','network_status','provider_risk_segment',
                     'avg_incomplete_submission_rate','region']],
              on='provider_id', how='left')
df = df.merge(memb[['member_id','age_band','plan_type','risk_level','chronic_condition_count']],
              on='member_id', how='left')
df = df.merge(svc[['service_id','service_category','procedure_group']],
              on='service_id', how='left')

df['overturned_flag'] = df['appeal_outcome'].isin(['Overturned','Partially Overturned']).astype(int)

print(f"Merged dataset: {df.shape}")
print(f"\nTarget: overturned_flag")
print(f"  Overturned/Partial (1): {df['overturned_flag'].sum()} ({df['overturned_flag'].mean()*100:.1f}%)")
print(f"  Upheld             (0): {(1-df['overturned_flag']).sum()} ({(1-df['overturned_flag'].mean())*100:.1f}%)")
print()
print("⚠ NOTE: 79.4% positive rate = inverted imbalance (majority class IS overturned)")
print("  A naive 'always predict overturned' baseline achieves 79.4% accuracy.")
print("  ROC-AUC measures discrimination — more meaningful than accuracy here.")


Merged dataset: (175, 31)

Target: overturned_flag
  Overturned/Partial (1): 139 (79.4%)
  Upheld             (0): 36 (20.6%)

⚠ NOTE: 79.4% positive rate = inverted imbalance (majority class IS overturned)
  A naive 'always predict overturned' baseline achieves 79.4% accuracy.
  ROC-AUC measures discrimination — more meaningful than accuracy here.


## 3. Feature Set

Includes `additional_documentation_submitted` — unique to appeal model.  
This feature is available at appeal intake (submitted with the appeal packet)  
and is not leakage.

All other features are the same pre-decision features used in Models 05 and 06.  
Note: `denial_reason` is still excluded — while available at this point in the  
workflow, including it would make the model useful only after the denial is issued,  
not before. The goal is pre-denial intervention.


In [ ]:
CAT_FEATURES  = ['request_type','submission_channel','provider_type','network_status',
                 'provider_risk_segment','plan_type','age_band','risk_level',
                 'service_category','region']
NUM_FEATURES  = ['estimated_cost','avg_incomplete_submission_rate','chronic_condition_count']
BOOL_FEATURES = ['documentation_complete','previous_denial_history',
                 'auto_eligible','clinical_review_required',
                 'additional_documentation_submitted']  # ← appeal-specific feature

for c in BOOL_FEATURES:
    df[c] = df[c].astype(str).map({'True':'1','False':'0','1':'1','0':'0'}).fillna('0').astype(int)

FEATURES = CAT_FEATURES + NUM_FEATURES + BOOL_FEATURES
df_model = df[FEATURES + ['overturned_flag']].dropna()

X = df_model[FEATURES]
y = df_model['overturned_flag']

print(f"Features: {len(FEATURES)} ({len(CAT_FEATURES)} cat, {len(NUM_FEATURES)} num, {len(BOOL_FEATURES)} bool)")
print(f"Rows after dropna: {len(df_model)}")
print(f"Positive rate: {y.mean():.3f}")


Features: 18 (10 cat, 3 num, 5 bool)
Rows after dropna: 175
Positive rate: 0.794


## 4. Preprocessing Pipeline

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('cat',  OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_FEATURES),
    ('num',  StandardScaler(), NUM_FEATURES),
    ('bool', 'passthrough', BOOL_FEATURES)
])
print("Preprocessing pipeline defined.")


Preprocessing pipeline defined.


## 5. 5-Fold Stratified Cross-Validation

With 175 rows and 79.4% positive rate, train/test split would leave ~35 rows in  
the test set (only ~7 upheld cases). Cross-validation provides more stable estimates.

Models: Logistic Regression and Decision Tree only.  
No GBM — complex models are not appropriate at this dataset size.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42,
                                               class_weight='balanced', C=1.0),
    'Decision Tree':       DecisionTreeClassifier(max_depth=3, random_state=42,
                                                   class_weight='balanced')
}

results = {}
for name, clf in models.items():
    pipe = Pipeline([('pre', preprocessor), ('clf', clf)])
    
    cv_results = cross_validate(
        pipe, X, y, cv=cv,
        scoring={'roc_auc': 'roc_auc', 'recall': 'recall',
                 'precision': 'precision', 'f1': 'f1'},
        return_train_score=False
    )
    
    results[name] = {
        'roc_auc_mean':    round(cv_results['test_roc_auc'].mean(), 4),
        'roc_auc_std':     round(cv_results['test_roc_auc'].std(), 4),
        'recall_mean':     round(cv_results['test_recall'].mean(), 4),
        'precision_mean':  round(cv_results['test_precision'].mean(), 4),
        'f1_mean':         round(cv_results['test_f1'].mean(), 4),
        'model': name,
        'target': 'overturned_flag',
        'n_rows': int(len(df_model)),
        'positive_rate': round(float(y.mean()), 4),
        'cv_folds': 5,
        'illustrative': True
    }
    
    print(f"{name} (5-fold CV):")
    print(f"  ROC-AUC : {results[name]['roc_auc_mean']:.4f} ± {results[name]['roc_auc_std']:.4f}")
    print(f"  Recall  : {results[name]['recall_mean']:.4f}")
    print(f"  Precision: {results[name]['precision_mean']:.4f}")
    print(f"  F1      : {results[name]['f1_mean']:.4f}")
    print()

print("Naive baseline (always predict overturned): ROC-AUC = 0.500, Recall = 1.0, Precision = 0.794")


Logistic Regression (5-fold CV):
  ROC-AUC : 0.6945 ± 0.0813
  Recall  : 0.7561
  Precision: 0.8545
  F1      : 0.8005

Decision Tree (5-fold CV):
  ROC-AUC : 0.6669 ± 0.0849
  Recall  : 0.6839
  Precision: 0.8716
  F1      : 0.7656

Naive baseline (always predict overturned): ROC-AUC = 0.500, Recall = 1.0, Precision = 0.794


## 6. Interpretation and Honest Assessment

### What the ROC-AUC means here
- Naive baseline: ROC-AUC = 0.50 (no discrimination)
- LR: ROC-AUC = 0.695 ± 0.081 — meaningful signal above baseline
- High standard deviation (±0.08) reflects the very small dataset

### The Precision/Recall Caution
- High precision (0.85) and recall (0.76) look strong, but the 79.4% base rate  
  means these scores are partially inflated by predicting the majority class correctly
- ROC-AUC is the more honest metric — it measures discrimination independent of threshold

### Most Important Honest Limitation
> **At 175 rows, confidence intervals are wide.** The ±0.081 std on ROC-AUC means  
> the true population ROC-AUC could plausibly range from ~0.61 to ~0.78.  
> This model is included as a **methodology demonstration** — the correct approach  
> for identifying appeal overturn risk — not as a deployable predictive model.

### Business Value of the Approach (if scaled)
If a real payer had 1,000+ appeal records, this model type could:
1. Flag newly denied cases where overturn probability exceeds a threshold (e.g., 0.80)
2. Route flagged cases to a medical director for pre-appeal review
3. Identify documentation patterns (via `additional_documentation_submitted` feature)  
   that predict overturn — informing initial denial process quality improvement

### Model Framing Statement
> This model does not approve or deny care. It is a methodology demonstration for  
> identifying appeal overturn risk. It would require substantially more data before  
> operational deployment. All data is synthetic. All claims are bound to this dataset.


## 7. Save Outputs

In [ ]:
metrics_list = list(results.values())
with open(OUT + 'appeal_metrics.json', 'w') as f:
    json.dump(metrics_list, f, indent=2)

print("✅ appeal_metrics.json saved.")
print(f"  {len(metrics_list)} model entries")
print(f"  illustrative=True flag included on all entries")


✅ appeal_metrics.json saved.
  2 model entries
  illustrative=True flag included on all entries
